In [11]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from dotenv import load_dotenv
from langchain_groq import ChatGroq
load_dotenv()

class LLMState(TypedDict):
    first_qa : str
    second_qa : str

In [12]:
def first(state: LLMState) -> LLMState:
    question = state['first_qa']
    state['first_qa'] = f'Answer this question: {question}'
    return state


In [15]:
def second(state: LLMState) -> LLMState:
    state['second_qa'] = state['first_qa'] + 'Translate it to telugu'

    model = ChatGroq(model="gpt-4o")

    response = model.invoke(state['second_qa'])

    state['second_qa'] = response.content

    return state

In [16]:
graph = StateGraph(LLMState)
graph.add_node('first', first)
graph.add_node('second', second)

graph.add_edge(START, 'first')
graph.add_edge('first', 'second')
graph.add_edge('second', END)

workflow = graph.compile()

output = workflow.invoke({'first_qa': 'What is the capital of France?'})

print(output)

NotFoundError: Error code: 404 - {'error': {'message': 'The model `gpt-4o` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}